<a href="https://colab.research.google.com/github/Mukundan-T/seqADAGE/blob/master/Py/muk_transfer_learning/genomic_mapping/Mukundan_sA_ec_pg_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E. coli &rightarrow; S. aureus Gene Mapping

### Mukundan Thanigaivelan

#### July 9, 2026

Here we are trying to map E. coli genes to S. aureus genes using techniques such as BLAST, k-mer mapping, and AlphaFold.

## 1. Connect to GitHub

In [45]:
!git clone https://github.com/Mukundan-T/seqADAGE.git

Cloning into 'seqADAGE'...
remote: Enumerating objects: 535, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 535 (delta 118), reused 49 (delta 16), pack-reused 323 (from 1)
Receiving objects: 100% (535/535), 46.74 MiB | 18.05 MiB/s, done.
Resolving deltas: 100% (271/271), done.


In [46]:
%cd seqADAGE/Py/muk_transfer_learning/genomic_mapping

/content/seqADAGE/Py/muk_transfer_learning/genomic_mapping/seqADAGE/Py/muk_transfer_learning/genomic_mapping


## 2. Loading classes & modules

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [49]:
# Data Analysis
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Miscellaneous
import time
import tensorflow as tf
import os

In [50]:
# check CPU and GPU available in runtime
print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU')))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

Num CPUs Available:  1
Num GPUs Available:  1
True


In [51]:
ec_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_pan_genome_reference.fa'
sa_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/sa_pan_genome_reference.fa'

## 3. BLAST

### Installation

In [ ]:
!apt-get -qq update
!apt-get -qq install ncbi-blast+

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ncbi-data.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.deb ...
Unpacking ncbi-blast+ (2.12.0+ds-3build1) ...
Setting up ncbi-data (6.1.20170106+dfsg1-9) ...
Setting up ncbi-blast+ (2.12.0+ds-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...


In [ ]:
!blastn -version

blastn: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


### Make BLAST database

In [ ]:
!makeblastdb -in "{ec_fasta}" -dbtype nucl -out ecoli_db



Building a new DB, current time: 07/10/2026 14:49:17
New DB name:   /content/seqADAGE/Py/muk_transfer_learning/genomic_mapping/ecoli_db
New DB title:  /content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_pan_genome_reference.fa
Sequence type: Nucleotide
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 68546 sequences in 3.00472 seconds.




### Blast S. aureus genes against database and save hits

In [ ]:
blast_hits_file = '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/blast_hits.tsv'

In [ ]:
!blastn \
  -query "{sa_fasta}" \
  -db ecoli_db \
  -out "{blast_hits_file}" \
  -outfmt 6

In [ ]:
columns = [
  "query", "subject", "pident", "length",
  "mismatch", "gapopen", "qstart", "qend",
  "sstart", "send", "evalue", "bitscore"
]

hits = pd.read_csv(blast_hits_file, sep = "\t", names = columns)
hits.shape

(7312, 12)

### Inspect and filter for best hits

In [ ]:
hits.head()

,query,subject,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,12b493b74dec297122c11824ed05ba01_3,panDB562_47409,99.780,1362,3,0,1,1362,1,1362,0.0,2499.0
1,12b493b74dec297122c11824ed05ba01_5,panDB562_47408,99.647,1134,4,0,1,1134,1,1134,0.0,2073.0
2,12b493b74dec297122c11824ed05ba01_9,panDB562_47407,99.820,1113,2,0,1,1113,1,1113,0.0,2045.0
3,12b493b74dec297122c11824ed05ba01_11,panDB562_47406,99.845,1935,3,0,1,1935,1,1935,0.0,3557.0
4,12b493b74dec297122c11824ed05ba01_13,panDB562_47405,98.390,2670,43,0,1,2670,1,2670,0.0,4693.0


In [ ]:
hits.describe()

,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
count,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7312.000000,7.312000e+03,7312.000000
mean,97.386560,548.622812,11.576586,0.509847,126.768326,674.059354,216.755607,717.489606,9.927411e-09,945.319283
std,4.403544,524.543432,28.895761,2.625259,1198.731851,1302.197932,425.747971,593.130040,4.479023e-07,940.864001
min,72.301000,28.000000,0.000000,0.000000,1.000000,28.000000,1.000000,1.000000,0.000000e+00,52.800000
25%,97.143000,209.000000,1.000000,0.000000,1.000000,227.750000,1.000000,300.000000,0.000000e+00,344.000000
50%,99.142000,357.000000,3.000000,0.000000,1.000000,402.000000,1.000000,602.000000,4.800000e-166,582.000000
75%,99.822000,759.000000,10.000000,0.000000,1.000000,858.000000,271.000000,975.000000,1.775000e-94,1308.000000
max,100.000000,7176.000000,415.000000,69.000000,31333.000000,31635.000000,7012.000000,7341.000000,3.750000e-05,13075.000000


In [ ]:
# Number of unique SA genes represented
hits['query'].nunique()

4356

In [ ]:
# Number of unique EC genes represented
hits['subject'].nunique()

5255

In [ ]:
# Match each SA gene to its top two best EC hits
best_hits = (
  hits
  .sort_values(["query", "pident"], ascending=[True, False])
  .groupby("query")
  .head(2)
)

best_hits.shape

(5091, 12)

In [ ]:
# Number of unique E. coli genes gathered
best_hits['subject'].nunique()

3224

In [ ]:
# Save best hits
best_hits.to_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_sa_blast_hits.csv',
  index = True
)

In [ ]:
# Verify I can load back in data
best_hits_loaded = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_sa_blast_hits.csv',
  index_col = 0
)
best_hits_loaded.shape

(5091, 12)

## 4. $k$-mer mapping

### Installation

In [52]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /usr/local/miniconda

ERROR: File or directory already exists: '/usr/local/miniconda'
If you want to update an existing installation, use the -u option.


In [53]:
# Add to PATH
os.environ["PATH"] += ":/usr/local/miniconda/bin"

In [54]:
!conda config --add channels defaults
!conda config --add channels bioconda
!conda config --add channels conda-forge

In [55]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


In [56]:
!conda install -y blat

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: - \ | done

# All requested packages already installed.



In [57]:
!blat

blat - Standalone BLAT v. 35 fast sequence search command line tool
usage:
   blat database query [-ooc=11.ooc] output.psl
where:
   database and query are each either a .fa , .nib or .2bit file,
   or a list these files one file name per line.
   -ooc=11.ooc tells the program to load over-occurring 11-mers from
               and external file.  This will increase the speed
               by a factor of 40 in many cases, but is not required
   output.psl is where to put the output.
   Subranges of nib and .2bit files may specified using the syntax:
      /path/file.nib:seqid:start-end
   or
      /path/file.2bit:seqid:start-end
   or
      /path/file.nib:start-end
   With the second form, a sequence id of file:start-end will be used.
options:
   -t=type     Database type.  Type is one of:
                 dna - DNA sequence
                 prot - protein sequence
                 dnax - DNA sequence translated in six frames to protein
               The default is dna
   -q=type     

### Run BLAT

In [58]:
# File to store k-mer best hits
kmer_hits_file = '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/kmer_hits.psl'

In [59]:
!blat \
  "{ec_fasta}" \
  "{sa_fasta}" \
  "{kmer_hits_file}"

Loaded 46157027 letters in 68546 sequences
Searched 6144633 bases in 9935 sequences


In [60]:
!head -20 "{kmer_hits_file}"

psLayout version 3

match	mis- 	rep. 	N's	Q gap	Q gap	T gap	T gap	strand	Q        	Q   	Q    	Q  	T        	T   	T    	T  	block	blockSizes 	qStarts	 tStarts
     	match	match	   	count	bases	count	bases	      	name     	size	start	end	name     	size	start	end	count
---------------------------------------------------------------------------------------------------------------------------------------------------------------
1359	3	0	0	0	0	0	0	+	12b493b74dec297122c11824ed05ba01_3	1362	0	1362	panDB562_47409	1362	0	1362	1	1362,	0,	0,
1130	4	0	0	0	0	0	0	+	12b493b74dec297122c11824ed05ba01_5	1134	0	1134	panDB562_47408	1134	0	1134	1	1134,	0,	0,
1111	2	0	0	0	0	0	0	+	12b493b74dec297122c11824ed05ba01_9	1113	0	1113	panDB562_47407	1113	0	1113	1	1113,	0,	0,
1932	3	0	0	0	0	0	0	+	12b493b74dec297122c11824ed05ba01_11	1935	0	1935	panDB562_47406	1935	0	1935	1	1935,	0,	0,
2617	42	0	0	1	7	1	1	+	12b493b74dec297122c11824ed05ba01_13	2670	0	2666	panDB562_47405	2670	0	2660	2	2650,9,	0,2657,	0,2651,
806	6	0	0	0	0

### Parse BLAT Output

In [61]:
psl_columns = [
  "matches", "misMatches", "repMatches", "nCount",
  "qNumInsert", "qBaseInsert", "tNumInsert", "tBaseInsert",
  "strand", "qName", "qSize", "qStart", "qEnd", "tName",
  "tSize", "tStart", "tEnd", "blockCount", "blockSizes",
  "qStarts", "tStarts",
]

blat_hits = pd.read_csv(
  kmer_hits_file,
  sep = "\t",
  skiprows = 5,
  names = psl_columns,
)
blat_hits.head()

,matches,misMatches,repMatches,nCount,qNumInsert,qBaseInsert,tNumInsert,tBaseInsert,strand,qName,...,qStart,qEnd,tName,tSize,tStart,tEnd,blockCount,blockSizes,qStarts,tStarts
0,1359,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_3,...,0,1362,panDB562_47409,1362,0,1362,1,"1362,","0,","0,"
1,1130,4,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_5,...,0,1134,panDB562_47408,1134,0,1134,1,"1134,","0,","0,"
2,1111,2,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_9,...,0,1113,panDB562_47407,1113,0,1113,1,"1113,","0,","0,"
3,1932,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_11,...,0,1935,panDB562_47406,1935,0,1935,1,"1935,","0,","0,"
4,2617,42,0,0,1,7,1,1,+,12b493b74dec297122c11824ed05ba01_13,...,0,2666,panDB562_47405,2670,0,2660,2,"2650,9,","0,2657,","0,2651,"


In [62]:
# Define a percentage match metric
blat_hits['pident'] = 100 * (blat_hits['matches'] / (blat_hits['matches'] + blat_hits['misMatches']))

# Custom score since we don't necessarily get a 'pident' --> reward high number of matches and high percentage of matches
blat_hits['score'] = blat_hits['matches'] * (blat_hits['pident'] / 100)
blat_hits.head()

,matches,misMatches,repMatches,nCount,qNumInsert,qBaseInsert,tNumInsert,tBaseInsert,strand,qName,...,tName,tSize,tStart,tEnd,blockCount,blockSizes,qStarts,tStarts,pident,score
0,1359,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_3,...,panDB562_47409,1362,0,1362,1,"1362,","0,","0,",99.779736,1356.006608
1,1130,4,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_5,...,panDB562_47408,1134,0,1134,1,"1134,","0,","0,",99.647266,1126.014109
2,1111,2,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_9,...,panDB562_47407,1113,0,1113,1,"1113,","0,","0,",99.820305,1109.003594
3,1932,3,0,0,0,0,0,0,+,12b493b74dec297122c11824ed05ba01_11,...,panDB562_47406,1935,0,1935,1,"1935,","0,","0,",99.844961,1929.004651
4,2617,42,0,0,1,7,1,1,+,12b493b74dec297122c11824ed05ba01_13,...,panDB562_47405,2670,0,2660,2,"2650,9,","0,2657,","0,2651,",98.420459,2575.663407


In [63]:
# Get top 2 E. coli hits for every S. aureus gene
top2_blat_hits = (
    blat_hits
    .sort_values(["qName", "score"], ascending=[True, False])
    .groupby("qName")
    .head(2)
)

top2_blat_hits.shape

(4914, 23)

In [64]:
# We've pulled about 3,200 E. coli genes
top2_blat_hits['tName'].nunique()

3199

In [65]:
# Save best hits
top2_blat_hits.to_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_sa_kmer_hits.csv',
  index = True
)

In [66]:
# Verify I can load back in data
blat_hits_loaded = pd.read_csv(
  '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/ec_sa_kmer_hits.csv',
  index_col = 0
)
blat_hits_loaded.shape

(4914, 23)

## 5. Foldseek

### Convert DNA FASTA to Protein FASTA

In [69]:
!pip install -qq biopython

In [70]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

def translate_fasta(input_fasta, output_fasta):
  proteins = []

  for record in SeqIO.parse(input_fasta, "fasta"):
    protein = record.seq.translate(to_stop=True)

    proteins.append(
      SeqRecord(
        protein,
        id = record.id,
        description = record.description
      )
    )

  SeqIO.write(proteins, output_fasta, "fasta")

In [72]:
translate_fasta(ec_fasta, "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_protein.faa")
translate_fasta(sa_fasta, "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/sa_protein.faa")

In [73]:
!head -10 "/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_protein.faa"

>11997cc26382b2c286cd502685a104a5_3 thrL
MKRISTTITTTITITTGNGAG
>11997cc26382b2c286cd502685a104a5_5 thrA
MRVLKFGGTSVANAERFLRVADILESNARQGQVATVLSAPAKITNHLVAMIEKTISGQDA
LPNISDAERIFAELLTGLAAAQPGFPLAQLKTFVDQEFAQIKHVLHGISLLGQCPDSINA
ALICRGEKMSIAIMAGVLEARGHNVTVIDPVEKLLAVGHYLESTVDIAESTRRIAASRIP
ADHMVLMAGFTAGNEKGELVVLGRNGSDYSAAVLAACLRADCCEIWTDVDGVYTCDPRQV
PDARLLKSMSYQEAMELSYFGAKVLHPRTITPIAQFQIPCLIKNTGNPQAPGTLIGASRD
EDELPVKGISNLNNMAMFSVSGPGMKGMVGMAARVFAAMSRARISVVLITQSSSEYSISF
CVPQSDCVRAERAMQEEFYLELKEGLLEPLAVTERLAIISVVGDGMRTLRGISAKFFAAL
